# DATA PREPROCESSING

In [34]:
from pathlib import Path
import pandas as pd
import json

### Dataset setup
- dataset downloaded from: https://www.clarin.si/repository/xmlui/handle/11356/2095
- experimenting with PISRS (Pravno-informacijski sistem Republike Slovenije) only for now

##### NOTES:
- JSONL = JSON Lines
    - each line = one document (one law, legal act...)
    - format: {"id": 1, "text": "..."
- PISRS (legislation)
    - each document contains: 
        - ("id", 128793), 
        - ("naziv", "Odredba o določitvi organizacij za opravljanje pregledov čolnov notranje plovbe"), 
        - ("mopedId", "ODRE1161"), 
        - ("eva", None), 
        - ("epa", None), 
        - ("sop", "1991-01-0419"), 
        - ("text", "Na podlagi petega odstavka 10. člena zakona ...")
    - each doc is structured by articles, like 1. člen ..., 2. člen ...




In [40]:
# move folders PISRS, SodnaPraksa, UradniList, USRS  into "/data"
DATA_DIR = Path("./data/PISRS")

# check for the jsonl files
jsonl_files = list(DATA_DIR.rglob("*.jsonl"))
print(len(jsonl_files))
for jsonl_file in jsonl_files[:10]:
    print(jsonl_file)
print()

# quick look at how they are structured
with open(jsonl_files[0], "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        doc = json.loads(line)
        print("doc: \n", doc.items(), "\n")
        print("doc text: \n", doc["text"][:1000])
        break

7
data/PISRS/obsoletni-in-konzumirani-predpisi.jsonl
data/PISRS/splosni-akti-za-izvrsevanje-javnih-pooblastil.jsonl
data/PISRS/neveljavni-predpisi.jsonl
data/PISRS/register-predpisov.jsonl
data/PISRS/evidenca-normodajalcev.jsonl
data/PISRS/drugi-splosni-in-posamicni-akti.jsonl
data/PISRS/predpisi-v-pripravi.jsonl

doc: 
 dict_items([('id', 128793), ('naziv', 'Odredba o določitvi organizacij za opravljanje pregledov čolnov notranje plovbe'), ('mopedId', 'ODRE1161'), ('eva', None), ('epa', None), ('sop', '1991-01-0419'), ('text', 'Na podlagi petega odstavka 10. člena zakona o varnosti pomorske in notranje plovbe (Uradni list SRS, št. 17/88) izdaja minister za promet in zveze\nODREDBO\no določitvi organizacij za opravljanje pregledov čolnov notranje plovbe\n1. člen\nPregled čolnov notranje plovbe opravljata:.\n-\xa0\xa0\xa0\xa0\xa0\xa0\xa0 Brodarsko društvo »SIDRO MIT«. Pot k čolnarni 42. Kamnica;\n-\xa0\xa0\xa0\xa0\xa0\xa0\xa0 »BI-TERMAL« Podjetje za proizvodnjo in posredovanje blaga na 

In [49]:
# print data for each doc
records = []
for file_path in jsonl_files:
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            doc = json.loads(line)
            text = doc.get("text", "")
            naziv = doc.get("naziv", "")
            records.append({
                "file_name": file_path.name,
                "id": doc.get("id"),
                "naziv": naziv,
                "mopedId": doc.get("mopedId"),
                "sop": doc.get("sop"),
                "text": text,
                "n_chars": len(text),
                "n_words": len(text.split())
            })

df = pd.DataFrame(records)
print(df.shape)
df.head()

(41084, 8)


,file_name,id,naziv,mopedId,sop,text,n_chars,n_words
0,obsoletni-in-konzumirani-predpisi.jsonl,128793,Odredba o določitvi organizacij za opravljanje...,ODRE1161,1991-01-0419,Na podlagi petega odstavka 10. člena zakona o ...,678,103
1,obsoletni-in-konzumirani-predpisi.jsonl,129125,Odredba o določitvi obrazca zahteve za uveljav...,ODRE610,1999-01-3696,Na podlagi petega odstavka 25. člena zakona o ...,833,121
2,obsoletni-in-konzumirani-predpisi.jsonl,129128,Pravilnik o postopku uveljavljanja pravic dela...,PRAV2087,1999-01-3784,Na podlagi četrtega odstavka 25. člena zakona ...,3128,450
3,obsoletni-in-konzumirani-predpisi.jsonl,129319,Odredba o enotnem zaščitnem znaku,ODRE310,1992-01-0337,Na podlagi 11. točke 18. člena zakona o promet...,2607,403
4,obsoletni-in-konzumirani-predpisi.jsonl,129828,Pravilnik o določitvi vrednosti točke za ugoto...,PRAV8223,2006-01-5917,Na podlagi 121. člena Stanovanjskega zakona (U...,1205,188


In [50]:
print("n_documents:", len(df))
print("n_files:", df["file_name"].nunique())
print("total_words:", df["n_words"].sum())
print("avg_words_per_doc:", df["n_words"].mean())
print("median_words_per_doc:", df["n_words"].median())
print("min_words_per_doc:", df["n_words"].min())
print("max_words_per_doc:", df["n_words"].max())

n_documents: 41084
n_files: 7
total_words: 97820603
avg_words_per_doc: 2380.990239509298
median_words_per_doc: 794.0
min_words_per_doc: 0
max_words_per_doc: 115850


In [51]:
grouped = df.groupby("file_name")
file_counts = grouped["id"].count().to_frame(name="n_documents")
file_counts["total_words"] = grouped["n_words"].sum()
file_counts["avg_words"] = grouped["n_words"].mean()
file_counts

,n_documents,total_words,avg_words
file_name,,,
drugi-splosni-in-posamicni-akti.jsonl,7198,15294944,2124.888024
evidenca-normodajalcev.jsonl,20629,49092992,2379.804741
neveljavni-predpisi.jsonl,2511,6159870,2453.154122
obsoletni-in-konzumirani-predpisi.jsonl,1072,850598,793.468284
predpisi-v-pripravi.jsonl,127,471597,3713.362205
register-predpisov.jsonl,7375,20091409,2724.258847
splosni-akti-za-izvrsevanje-javnih-pooblastil.jsonl,2172,5859193,2697.602670
